# 01 — Statistical & Temporal Feature Extraction

Extract per-band statistical and temporal features for every training object.
These features, combined with periodicity features (next notebook), form the
input to the Random Forest baseline.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sys.path.insert(0, os.path.abspath("../.."))
from config import DATA_CONFIG
from src.data import get_object_lightcurve
from src.features import extract_all_features, extract_per_band_features
from src.utils import save_figure

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
metadata = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_metadata.parquet'))
lc = pd.read_parquet(os.path.join(DATA_CONFIG['processed_dir'], 'training_lc.parquet'))
print(f"Loaded {len(metadata)} objects, {len(lc):,} observations")

## 1. Feature Extraction Demo — Single Object

Demonstrate the full feature extraction pipeline on one example object.

In [ ]:
# Pick one SN Ia (class 90) for demo
demo_id = metadata[metadata['target'] == 90]['object_id'].iloc[0]
demo_lc = get_object_lightcurve(demo_id, lc)
demo_meta = metadata[metadata['object_id'] == demo_id].iloc[0]

print(f"Demo object: {demo_id} (SN Ia)")
print(f"Bands available: {list(demo_lc.keys())}")
for band, data in demo_lc.items():
    print(f"  Band {DATA_CONFIG['bands'][band]}: {len(data['mjd'])} observations")

In [ ]:
# Extract all features
demo_features = extract_all_features(demo_id, demo_lc, demo_meta)
print(f"Total features: {len(demo_features)}")
print(f"\nSample features:")
for k, v in list(demo_features.items())[:20]:
    print(f"  {k:35s}: {v}")

## 2. Extract Features for All Training Objects

In [ ]:
# This takes a few minutes on ~7800 objects
all_features = []
failed = []

for _, row in tqdm(metadata.iterrows(), total=len(metadata), desc="Extracting features"):
    obj_id = row['object_id']
    try:
        object_lc = get_object_lightcurve(obj_id, lc)
        features = extract_all_features(obj_id, object_lc, row)
        features['target'] = row['target']
        all_features.append(features)
    except Exception as e:
        failed.append((obj_id, str(e)))

feature_df = pd.DataFrame(all_features)
print(f"\nExtracted features for {len(feature_df)} objects")
print(f"Failed: {len(failed)}")
print(f"Feature columns: {len(feature_df.columns)}")
if failed:
    print(f"First few failures: {failed[:5]}")

## 3. Feature Table Inspection

In [ ]:
print(f"Shape: {feature_df.shape}")
print(f"\nNaN count per column (top 20):")
nan_counts = feature_df.isnull().sum().sort_values(ascending=False)
print(nan_counts[nan_counts > 0].head(20))

print(f"\nTotal NaN fraction: {feature_df.isnull().sum().sum() / feature_df.size:.3f}")

In [ ]:
# Descriptive statistics
feature_df.describe().T.head(20)

## 4. Feature Correlations

In [ ]:
# Correlation matrix for g-band features (representative)
g_cols = [c for c in feature_df.columns if c.startswith('g_') and feature_df[c].dtype in ['float64', 'int64']]
corr = feature_df[g_cols].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, center=0, cmap='RdBu_r', square=True, ax=ax,
            xticklabels=True, yticklabels=True)
ax.set_title('Feature Correlations (g-band)')
plt.xticks(fontsize=7, rotation=90)
plt.yticks(fontsize=7)
plt.tight_layout()
save_figure(fig, 'feature_correlations_g')
plt.show()

## 5. Save Feature Table

Save without periodicity features — those are added in the next notebook.

In [ ]:
out_path = os.path.join(DATA_CONFIG['processed_dir'], 'features_statistical.parquet')
feature_df.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({os.path.getsize(out_path)/1e6:.1f} MB)")